# arg-position-back-functions — ex2: write pow_back0 and pow_back1 — polynomial vs log-exponential per-arg back fns

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `arg-position-back-functions`. Running the final beacon cell reports progress against the `Backprop: Arg-position back funcs` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: Arg-position back funcs` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`arg-position-back-functions`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "arg-position-back-functions"
DD_SUBTOPIC = "Backprop: Arg-position back funcs"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Arg-position back fns — quick refresher

Binary ops register **two** back fns — one per input position — because the gradient w.r.t. each arg is a different function of `(grad_out, x, y)`.

**Worked exemplar.** For `out = x ** y` (element-wise power):
```
d(x**y)/dx = y * x**(y-1)             # 'pow_back0', for arg-0 (x)
d(x**y)/dy = x**y * log(x)            # 'pow_back1', for arg-1 (y)
```
Both back fns take `(grad_out, out, x, y)`. `pow_back0` reuses `x` and `y`; `pow_back1` reuses the cached `out` (= `x**y`) and `log(x)`.

### Exercise 2 — write pow_back0 and pow_back1 — polynomial vs log-exponential per-arg back fns

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the (grad_out, out, x, y) back-fn convention to write pow_back0 (polynomial form) and pow_back1 (log-exponential form) for out = x ** y, picking the right cached value in each.
> Keywords: pow-back, binary-op, log-derivative, per-arg-back-fn
> ```

**KCs targeted:** `arg-position-back-functions`, `back-fn-uses-cached-out`

Implement TWO back fns for the elementwise power op `out = x ** y` (`x > 0` so logs are defined).

**1. `pow_back0(grad_out, out, x, y) -> grad_x`** — gradient w.r.t. `x`.
   Math: `d(x**y)/dx = y * x**(y-1)`.
   Return `grad_out * y * x ** (y - 1)`, shape == `x.shape`.

**2. `pow_back1(grad_out, out, x, y) -> grad_y`** — gradient w.r.t. `y`.
   Math: `d(x**y)/dy = x**y * log(x) = out * log(x)`.
   Return `grad_out * out * t.log(x)`, shape == `y.shape`. Reuse the cached `out`, do not recompute `x ** y`.

The test checks scalar, vector, and matrix shapes, and cross-checks against `torch.autograd` on a `requires_grad=True` ground truth.

In [ ]:
def pow_back0(grad_out: Tensor, out: Tensor, x: Tensor, y: Tensor) -> Tensor:
    """dL/dx for out = x ** y (x > 0)."""
    raise NotImplementedError()


def pow_back1(grad_out: Tensor, out: Tensor, x: Tensor, y: Tensor) -> Tensor:
    """dL/dy for out = x ** y (x > 0)."""
    raise NotImplementedError()


def _test_ex2():
    # --- scalar sanity: 2**3 = 8 ---
    x = t.tensor([2.0]); y = t.tensor([3.0])
    out = x ** y
    g0 = pow_back0(t.tensor([1.0]), out, x, y)
    g1 = pow_back1(t.tensor([1.0]), out, x, y)
    # d/dx (x**3) at x=2 = 3*4 = 12;  d/dy (2**y) at y=3 = 8 * ln(2)
    assert t.allclose(g0, t.tensor([12.0])), f'pow_back0 scalar: {g0}'
    assert t.allclose(g1, t.tensor([8.0 * float(t.log(t.tensor(2.0)))])), (
        f'pow_back1 scalar: {g1}'
    )

    # --- vector ---
    x = t.tensor([1.5, 2.0, 4.0])
    y = t.tensor([2.0, 3.0, 0.5])
    out = x ** y
    grad_out = t.tensor([1.0, 1.0, 1.0])
    g0 = pow_back0(grad_out, out, x, y)
    g1 = pow_back1(grad_out, out, x, y)
    assert g0.shape == x.shape, f'pow_back0 shape: {g0.shape}'
    assert g1.shape == y.shape, f'pow_back1 shape: {g1.shape}'
    assert t.allclose(g0, y * x ** (y - 1)), f'pow_back0 vector: {g0}'
    assert t.allclose(g1, out * t.log(x)), f'pow_back1 vector: {g1}'

    # --- non-unit grad_out, matrix ---
    rng = t.Generator().manual_seed(7)
    X = t.rand(3, 4, generator=rng) * 2 + 0.5    # positive
    Y = t.rand(3, 4, generator=rng) * 2 + 0.5
    OUT = X ** Y
    G = t.randn(3, 4, generator=rng)
    g0 = pow_back0(G, OUT, X, Y)
    g1 = pow_back1(G, OUT, X, Y)
    assert g0.shape == X.shape and g1.shape == Y.shape
    assert t.allclose(g0, G * Y * X ** (Y - 1)), 'pow_back0 matrix mismatch'
    assert t.allclose(g1, G * OUT * t.log(X)), 'pow_back1 matrix mismatch'

    # --- cross-check vs autograd ---
    xa = t.tensor([1.5, 2.0, 4.0], requires_grad=True)
    ya = t.tensor([2.0, 3.0, 0.5], requires_grad=True)
    loss = (xa ** ya).sum()
    loss.backward()
    g0_ref = xa.grad
    g1_ref = ya.grad
    out_ref = (xa ** ya).detach()
    g0_ours = pow_back0(t.ones_like(out_ref), out_ref, xa.detach(), ya.detach())
    g1_ours = pow_back1(t.ones_like(out_ref), out_ref, xa.detach(), ya.detach())
    assert t.allclose(g0_ours, g0_ref, atol=1e-5), (
        f'pow_back0 vs autograd: ours={g0_ours} ref={g0_ref}'
    )
    assert t.allclose(g1_ours, g1_ref, atol=1e-5), (
        f'pow_back1 vs autograd: ours={g1_ours} ref={g1_ref}'
    )
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
def pow_back0(grad_out, out, x, y):
    return grad_out * y * x ** (y - 1)


def pow_back1(grad_out, out, x, y):
    return grad_out * out * t.log(x)
```

**Why pow_back1 reuses `out`.** `out == x ** y` is already cached; recomputing `x ** y` inside the back fn is wasteful and risks numerical drift. The `(grad_out, out, x, y)` signature exists precisely so back fns can lean on the forward's cached result.

**Why pow_back0 does NOT need `out`.** The derivative `y * x**(y-1)` is a function of the inputs alone — `out` is unused here. Both back fns receive `out` for uniform dispatch even when one doesn't need it.

**Domain restriction.** `log(x)` is undefined for `x <= 0`, so this back fn assumes `x > 0` at call time. Real-world autograd implementations either restrict the domain or use the complex-log extension; for ARENA we keep `x` positive in the tests.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()